# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")
# Display keywords, if available
if hasattr(metadata, 'keywords'):
    print("Keywords:", metadata.keywords)
# Show dataset DOI and version
if hasattr(metadata, 'identifier'):
    print(f"DOI: {metadata.identifier}")
if hasattr(metadata, 'version'):
    print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, their `@id`s, and fields. All Croissant entities are referenced by their `@id` for consistency. Below, we show which record sets, their fields (columns), and sample records are available.

In [ ]:
import itertools

# List all record sets and their @id
print("Available Record Sets:")
record_sets = list(dataset.record_sets)

for i, rs in enumerate(record_sets):
    print(f"  {i+1}. @id: {rs.id}  (name: {getattr(rs, 'name', None)})")

# We'll explore the first record set in detail
if record_sets:
    selected_record_set = record_sets[0]
    selected_record_set_id = selected_record_set.id
    print(f"\nSelected RecordSet @id: {selected_record_set_id}")
    # List available columns/fields
    print("Fields (@id):")
    for field in selected_record_set.fields:
        print(f"  - {field.id}")
    # Show a couple of records
    print("\nSample records from this RecordSet:")
    example_records = list(itertools.islice(dataset.records(record_set=selected_record_set_id), 3))
    if example_records:
        pprint.pprint(example_records)
    else:
        print("  No data records available.")
else:
    selected_record_set_id = None
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from the available record set into a DataFrame for analysis using the record set and field `@id`s.

This approach loads all available record sets (by their `@id`) into individual DataFrames indexed by `@id` for convenient processing.

In [ ]:
# Extract data from all record sets into a dictionary of DataFrames, by @id
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

print(f"Loading record sets: {record_set_ids}\n")
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[record_set_id] = df
    print(f"RecordSet @id: {record_set_id} -> shape: {df.shape}")

# Show the first 5 columns and records of the primary record set
if selected_record_set_id in dataframes:
    print("\nColumns:")
    print(dataframes[selected_record_set_id].columns.tolist())
    print("\nFirst 5 rows:")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Here, we demonstrate filtering and transforming numeric fields, grouped aggregations, and other processing.

You can update the `numeric_field` and `group_field` variables based on the columns identified above. All field references use their record set `@id` and column (field) name.

> **Note:** If unsure of the numeric fields, check column names and their types in the dataset documentation or sample data.

In [ ]:
# User: Update these variables if necessary

# Example choice: assume the record set contains 'cr:MW' (Molecular Weight) and 'cr:Description' fields.
main_rs_id = selected_record_set_id
main_df = dataframes[main_rs_id]

# Try to find a numeric column as example
if not main_df.empty:
    # Heuristics: Look for a float/integer column
    numeric_cols = main_df.select_dtypes(include=['number']).columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]  # Pick the first numeric column
    else:
        # Fallback: try common names
        for candidate in ['cr:MW', 'cr:Coverage', 'cr:Peptide_count']:
            if candidate in main_df.columns:
                numeric_field = candidate
                break
        else:
            numeric_field = None
    # For grouping, try to use a string column
    string_cols = main_df.select_dtypes(include=['object']).columns.tolist()
    group_field = None
    for candidate in ['cr:Description', 'cr:Accession'] + string_cols:
        if candidate in main_df.columns and candidate != numeric_field:
            group_field = candidate
            break
else:
    numeric_field = None
    group_field = None

print(f"Using RecordSet @id: {main_rs_id}")
print(f"Numeric field chosen: {numeric_field}")
print(f"Group field chosen: {group_field}\n")

if numeric_field and not main_df[numeric_field].empty:
    threshold = main_df[numeric_field].mean() if main_df[numeric_field].mean() > 0 else 10
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (N={len(filtered_df)}):\n")
    display(filtered_df.head())

    # Normalize the chosen numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field if available and show means
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA or dataframe is empty.")

## 5. Visualization
Let's visualize the distribution of a selected numeric field and, if possible, show its grouping.

Below, we plot a histogram of the normalized numeric field and a boxplot grouped by the grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not main_df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group_field if appropriate
    if group_field and group_field in main_df.columns:
        grp_sample = main_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        top_groups = grp_sample.index.tolist()
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=main_df[main_df[group_field].isin(top_groups)], palette="Set2")
        plt.title(f"{numeric_field} grouped by {group_field} (top 10 groups)")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("Cannot visualize. No suitable numeric field or dataframe is empty.")

## 6. Conclusion

In this notebook, we loaded the FAIR^2 Croissant dataset for mass spectrometry analysis of extracellular vesicles from stimulated human mast cells, using the `mlcroissant` library. Using only entity `@id` fields, we've demonstrated how to:

- Discover available record sets and fields
- Extract and preview tabular data
- Filter, normalize, and group data by numeric and categorical columns
- Visualize numeric distributions and group statistics

This approach ensures robust, standards-compliant dataset exploration for downstream analysis and FAIR data workflows.